# Final Code


In [1]:
!pip install -q ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 34.0 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
from ultralytics import YOLO
import os

# 1. Load the model
model = YOLO("yolo11s-seg.pt")

# 2. Start Training
model.train(
    data='/content/drive/MyDrive/FYP/climbing_dataset/data.yaml',
    epochs=500,
    imgsz=960,
    batch=8,             # Use AutoBatch to avoid the memory crash
    patience=0,           # Runs all 500 epochs
    lr0=0.001,            # Stable learning rate
    optimizer="AdamW",
    # Save directly to your Google Drive so results are permanent
    project="/content/drive/MyDrive/FYP/Training_Results",
    name="climbing_v1_960px",
    device=0,
    save=True,
    plots=True,
    deterministic=True
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.3.248 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/FYP/climbing_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=F

ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a6949f257c0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.04104

# Image Inference

In [8]:
import cv2
import numpy as np
import glob
import os
import matplotlib.pyplot as plt
from ultralytics import YOLO
from shapely.geometry import Polygon

# -----------------------------
# 0. CONFIGURATION
# -----------------------------
RES_NAME = "960"
IMGSZ = int(RES_NAME)
DISTANCE_THRESHOLD = 1  # pixels (cropped / resized images)
THRESH_NAME = f"threshold_{DISTANCE_THRESHOLD}"

# -----------------------------
# 1. Load Models
# -----------------------------
person_model = YOLO("/content/drive/MyDrive/FYP/models/yolov8n.pt")
equip_model = YOLO("/content/drive/MyDrive/FYP/models/best_960.pt")

# -----------------------------
# 2. Paths
# -----------------------------
input_folder = "/content/drive/MyDrive/FYP/test_images/"
base_output_folder = os.path.join("/content/drive/MyDrive/FYP/inference/", RES_NAME)
# New folder for threshold-specific results
output_folder = os.path.join(base_output_folder, THRESH_NAME)
os.makedirs(output_folder, exist_ok=True)

image_files = sorted(
    glob.glob(os.path.join(input_folder, "*.jpg")) +
    glob.glob(os.path.join(input_folder, "*.JPG"))
)[:6]

# -----------------------------
# 3. Helper Functions (UNIFIED)
# -----------------------------
def mask_to_polygon(mask, w, h):
    mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_LINEAR)
    mask = (mask * 255).astype(np.uint8)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    cnt = max(contours, key=cv2.contourArea)
    if len(cnt) < 3:
        return None
    return cnt, Polygon(cnt.squeeze())

# -----------------------------
# 4. Image Processing
# -----------------------------
def process_image(img_path):
    frame = cv2.imread(img_path)
    if frame is None:
        return None

    h, w = frame.shape[:2]
    status = {"Person": False, "Rope": False, "Harness": False, "Climber": False}

    # --- Person Detection ---
    p_result = person_model(frame, verbose=False)[0]
    person_poly = None
    person_bbox = None

    for box in p_result.boxes:
        if int(box.cls[0]) == 0 and float(box.conf[0]) > 0.5:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            person_bbox = (x1, y1, x2, y2)
            person_poly = Polygon([(x1,y1),(x2,y1),(x2,y2),(x1,y2)])
            status["Person"] = True
            break

    if not status["Person"]:
        return frame

    # --- Equipment Detection ---
    e_result = equip_model(frame, imgsz=IMGSZ, verbose=False)[0]
    ropes, harnesses = [], []

    if e_result.masks is not None:
        for i, cls_id in enumerate(e_result.boxes.cls):
            if float(e_result.boxes.conf[i]) < 0.4:
                continue

            res = mask_to_polygon(e_result.masks.data[i].cpu().numpy(), w, h)
            if res is None:
                continue

            cnt, poly = res

            # Person association via Shapely
            if not person_poly.intersects(poly):
                continue

            if int(cls_id) == 0:
                harnesses.append((cnt, poly))
                status["Harness"] = True
            elif int(cls_id) == 1:
                ropes.append((cnt, poly))
                status["Rope"] = True

    # --- Rope–Harness Attachment Logic ---
    for _, r_poly in ropes:
        for _, h_poly in harnesses:
            if r_poly.intersects(h_poly) or r_poly.distance(h_poly) <= DISTANCE_THRESHOLD:
                status["Climber"] = True
                break

    # --- Drawing ---
    for cnt, _ in ropes:
        cv2.polylines(frame, [cnt], True, (0,255,0), 3)
    for cnt, _ in harnesses:
        cv2.polylines(frame, [cnt], True, (0,165,255), 3)

    color = (0,255,0) if status["Climber"] else (0,0,255)
    cv2.rectangle(frame, person_bbox[:2], person_bbox[2:], color, 4)

    y = 50
    for k, v in status.items():
        cv2.putText(frame, f"{k}: {'OK' if v else 'MISSING'}",
                    (30, y), cv2.FONT_HERSHEY_SIMPLEX, 1.1,
                    (0,255,0) if v else (0,0,255), 3)
        y += 55

    return frame

# -----------------------------
# 5. Run Batch
# -----------------------------
processed = []
for path in image_files:
    img = process_image(path)
    if img is not None:
        # Save inside threshold-specific folder
        out_path = os.path.join(output_folder, f"inferenced_{os.path.basename(path)}")
        cv2.imwrite(out_path, img)
        processed.append(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        print(f"Saved: {out_path}")

# Display as grid
plt.figure(figsize=(20,12))
for i, img in enumerate(processed):
    plt.subplot(2,3,i+1)
    plt.imshow(img)
    plt.axis("off")
plt.tight_layout()

# Save grid
grid_path = os.path.join(output_folder, f"grid_{THRESH_NAME}.png")
plt.savefig(grid_path, dpi=300)
print(f"Grid saved: {grid_path}")
plt.show()


Output hidden; open in https://colab.research.google.com to view.

# Video Inference

In [17]:
import cv2
import numpy as np
import os
from ultralytics import YOLO
from shapely.geometry import Polygon

# -----------------------------
# 0. CONFIGURATION
# -----------------------------
RES_NAME = "960"
IMGSZ = int(RES_NAME)
DISTANCE_THRESHOLD = 1  # pixels (1080p video noise)
THRESH_NAME = f"threshold_{DISTANCE_THRESHOLD}"

# -----------------------------
# 1. Load Models
# -----------------------------
person_model = YOLO("/content/drive/MyDrive/FYP/models/yolov8n.pt")
equip_model = YOLO("/content/drive/MyDrive/FYP/models/best_960.pt")

# -----------------------------
# 2. Paths
# -----------------------------
input_video = "/content/drive/MyDrive/FYP/test/test1.mp4"
base_output_dir = os.path.join("/content/drive/MyDrive/FYP/inference/", RES_NAME)
output_dir = os.path.join(base_output_dir, THRESH_NAME)
os.makedirs(output_dir, exist_ok=True)

output_video = os.path.join(output_dir, "test1_inferenced.mp4")

# -----------------------------
# 3. Video Setup
# -----------------------------
cap = cv2.VideoCapture(input_video)
fps = int(cap.get(cv2.CAP_PROP_FPS))
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter(output_video, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

# -----------------------------
# 4. Helper Function
# -----------------------------
def mask_to_polygon(mask):
    mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_LINEAR)
    mask = (mask * 255).astype(np.uint8)
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    cnt = max(cnts, key=cv2.contourArea)
    if len(cnt) < 3:
        return None
    return cnt, Polygon(cnt.squeeze())

# -----------------------------
# 5. Main Loop
# -----------------------------
while True:
    ret, frame = cap.read()
    if not ret:
        break

    status = {"Person": False, "Rope": False, "Harness": False, "Climber": False}

    # --- Person Detection ---
    p_result = person_model(frame, verbose=False)[0]
    person_poly = None
    person_bbox = None

    for box in p_result.boxes:
        if int(box.cls[0]) == 0 and float(box.conf[0]) > 0.5:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            person_bbox = (x1, y1, x2, y2)
            person_poly = Polygon([(x1, y1), (x2, y1), (x2, y2), (x1, y2)])
            status["Person"] = True
            break

    if status["Person"]:
        # --- Equipment Detection ---
        e_result = equip_model(frame, imgsz=IMGSZ, verbose=False)[0]
        ropes, harnesses = [], []

        if e_result.masks is not None:
            for i, cls_id in enumerate(e_result.boxes.cls):
                if float(e_result.boxes.conf[i]) < 0.4:
                    continue

                res = mask_to_polygon(e_result.masks.data[i].cpu().numpy())
                if res is None:
                    continue

                cnt, poly = res

                # Person association via Shapely
                if not person_poly.intersects(poly):
                    continue

                if int(cls_id) == 0:
                    harnesses.append((cnt, poly))
                    status["Harness"] = True
                elif int(cls_id) == 1:
                    ropes.append((cnt, poly))
                    status["Rope"] = True

        # --- Rope–Harness attachment logic ---
        for _, r_poly in ropes:
            for _, h_poly in harnesses:
                if r_poly.intersects(h_poly) or r_poly.distance(h_poly) <= DISTANCE_THRESHOLD:
                    status["Climber"] = True
                    break

        # --- Draw Polygons and Bounding Box ---
        for cnt, _ in ropes:
            cv2.polylines(frame, [cnt], True, (0, 255, 0), 3)
        for cnt, _ in harnesses:
            cv2.polylines(frame, [cnt], True, (0, 165, 255), 3)

        box_color = (0, 255, 0) if status["Climber"] else (0, 0, 255)
        cv2.rectangle(frame, person_bbox[:2], person_bbox[2:], box_color, 4)

        # --- Draw Status Labels ---
        y0 = 50
        for k, v in status.items():
            txt_color = (0, 255, 0) if v else (0, 0, 255)
            cv2.putText(frame, f"{k}: {'OK' if v else 'MISSING'}",
                        (30, y0), cv2.FONT_HERSHEY_SIMPLEX, 1.0, txt_color, 3)
            y0 += 55

    out.write(frame)

cap.release()
out.release()
print("Video saved:", output_video)


Video saved: /content/drive/MyDrive/FYP/inference/960/threshold_1/test1_inferenced.mp4


# Model Stats

In [18]:
import pandas as pd
import plotly.graph_objects as go

# Load the CSV
df = pd.read_csv("/content/drive/MyDrive/FYP/Training_Results/climbing_v1_960/results.csv")

# ----------------------------
# Combined Beautiful Radar Chart
# ----------------------------
fig = go.Figure()

# 1. Metrics for class B
categories_B = ['<b>Precision(B)</b>', '<b>Recall(B)</b>', '<b>mAP50(B)</b>', '<b>mAP50-95(B)</b>']
values_B = df.iloc[-1][['metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)']].tolist()

fig.add_trace(go.Scatterpolar(
    r=values_B,
    theta=categories_B,
    fill='toself',
    fillcolor='rgba(99, 110, 250, 0.3)',
    line=dict(color='rgba(99, 110, 250, 0.6)', width=2),  # 60% opacity
    marker=dict(size=10, color='rgba(99, 110, 250, 0.6)'),  # 60% opacity
    name='<b>BBox</b>',
    opacity=0.8,
    text=[f'{v:.3f}' for v in values_B],
    textposition='top center',
    textfont=dict(size=10, color='rgba(99, 110, 250, 1)', family='Arial Black'),  # Full opacity
    mode='lines+markers+text',
    hovertemplate='<b>%{theta}</b><br>Value: %{r:.3f}<extra></extra>'
))

# 2. Metrics for class M
categories_M = ['<b>Precision(M)</b>', '<b>Recall(M)</b>', '<b>mAP50(M)</b>', '<b>mAP50-95(M)</b>']
values_M = df.iloc[-1][['metrics/precision(M)', 'metrics/recall(M)', 'metrics/mAP50(M)', 'metrics/mAP50-95(M)']].tolist()

fig.add_trace(go.Scatterpolar(
    r=values_M,
    theta=categories_M,
    fill='toself',
    fillcolor='rgba(239, 85, 59, 0.3)',
    line=dict(color='rgba(239, 85, 59, 0.6)', width=2),  # 60% opacity
    marker=dict(size=10, color='rgba(239, 85, 59, 0.6)'),  # 60% opacity
    name='<b>Mask</b>',
    opacity=0.8,
    text=[f'{v:.3f}' for v in values_M],
    textposition='top center',
    textfont=dict(size=10, color='rgba(239, 85, 59, 1)', family='Arial Black'),  # Full opacity
    mode='lines+markers+text',
    hovertemplate='<b>%{theta}</b><br>Value: %{r:.3f}<extra></extra>'
))

# 3. Train losses
categories_train = ['<b>Box Loss(T)</b>', '<b>Seg Loss(T)</b>', '<b>Cls Loss(T)</b>', '<b>DFL Loss(T)</b>']
values_train = df.iloc[-1][['train/box_loss', 'train/seg_loss', 'train/cls_loss', 'train/dfl_loss']].tolist()

fig.add_trace(go.Scatterpolar(
    r=values_train,
    theta=categories_train,
    fill='toself',
    fillcolor='rgba(0, 204, 150, 0.3)',
    line=dict(color='rgba(0, 204, 150, 0.6)', width=2),  # 60% opacity
    marker=dict(size=10, color='rgba(0, 204, 150, 0.6)'),  # 60% opacity
    name='<b>Train Loss</b>',
    opacity=0.8,
    text=[f'{v:.3f}' for v in values_train],
    textposition='top center',
    textfont=dict(size=10, color='rgba(0, 204, 150, 1)', family='Arial Black'),  # Full opacity
    mode='lines+markers+text',
    hovertemplate='<b>%{theta}</b><br>Value: %{r:.3f}<extra></extra>'
))

# 4. Validation losses
categories_val = ['<b>Box Loss(V)</b>', '<b>Seg Loss(V)</b>', '<b>Cls Loss(V)</b>', '<b>DFL Loss(V)</b>']
values_val = df.iloc[-1][['val/box_loss', 'val/seg_loss', 'val/cls_loss', 'val/dfl_loss']].tolist()

fig.add_trace(go.Scatterpolar(
    r=values_val,
    theta=categories_val,
    fill='toself',
    fillcolor='rgba(255, 161, 90, 0.3)',
    line=dict(color='rgba(255, 161, 90, 0.6)', width=2),  # 60% opacity
    marker=dict(size=10, color='rgba(255, 161, 90, 0.6)'),  # 60% opacity
    name='<b>Val Loss</b>',
    opacity=0.8,
    text=[f'{v:.3f}' for v in values_val],
    textposition='top center',
    textfont=dict(size=10, color='rgba(255, 161, 90, 1)', family='Arial Black'),  # Full opacity
    mode='lines+markers+text',
    hovertemplate='<b>%{theta}</b><br>Value: %{r:.3f}<extra></extra>'
))

# Layout
fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            showticklabels=False,
            gridcolor='rgba(128, 128, 128, 0.2)'
        ),
        angularaxis=dict(
            gridcolor='rgba(128, 128, 128, 0.2)',
            tickfont=dict(size=12, color='#2c3e50'),
            rotation=45
        ),
        bgcolor='rgba(51, 51, 51, 0.1)',
        domain=dict(x=[0, 1], y=[0, 0.88])
    ),
    showlegend=True,
    legend=dict(
        orientation='h',
        x=0.5,
        y=1.05,
        xanchor='center',
        yanchor='bottom',
        font=dict(size=12, color='#2c3e50'),
        bgcolor='rgba(255, 255, 255, 0.8)',
        bordercolor='rgba(128, 128, 128, 0.3)',
        borderwidth=0.5,
        itemsizing='constant',
        tracegroupgap=5,
        itemwidth=30,
        entrywidth=70,
        entrywidthmode='pixels'
    ),
    # title=dict(
    #     text='🎯 Model Performance',
    #     font=dict(size=14, color='#2c3e50'),
    #     x=0.5,
    #     xanchor='center',
    #     y=0.96,
    #     yanchor='top'
    # ),
    paper_bgcolor='white',
    height=600,
    width=700
)
# fig.write_image("radar_chart.png", width=900, height=700, scale=3)
fig.show()